In [ ]:
import os
import pandas as pd
from statsmodels.tsa.stattools import adfuller

In [10]:
# Load transformed dataset
df = pd.read_csv("../data/eda_transformed.csv")
df

,Stocks,Date,Open,High,Low,Volume,Close
0,Apple,2005-07-25,1.322084,1.330800,1.314270,294627200,1.316674
1,Apple,2005-07-26,1.322685,1.325690,1.303149,268592800,1.311264
2,Apple,2005-07-27,1.317275,1.324488,1.282412,283749200,1.322083
3,Apple,2005-07-28,1.317876,1.322385,1.301347,251311200,1.316374
4,Apple,2005-07-29,1.309160,1.333805,1.270090,562080400,1.281811
...,...,...,...,...,...,...,...
25135,Johnson&Johnson,2025-07-14,156.869995,157.470001,155.520004,10185600,156.820007
25136,Johnson&Johnson,2025-07-15,156.360001,157.190002,154.800003,6873200,155.169998
25137,Johnson&Johnson,2025-07-16,160.300003,166.119995,159.800003,22134800,164.779999
25138,Johnson&Johnson,2025-07-17,163.179993,164.699997,162.300003,11295700,162.979996


In [11]:
# Convert 'Date' to datetime
df['Date'] = pd.to_datetime(df['Date'])

# Stocks mapping
print("🔄 Defining stock ticker mappings...")
stocks = {
    "Apple": "Apple",
    "GeneralElectric": "GeneralElectric",
    "IBM": "IBM",
    "Johnson&Johnson": "JohnsonJohnson",  # Removing special char for file name
    "Microsoft": "Microsoft"
}
print("✅ Stocks mapping ready.\n")

🔄 Defining stock ticker mappings...
✅ Stocks mapping ready.



In [12]:
# Create output directory
output_dir = "../data/ARIMA"
print(f"📁 Creating output directory: {output_dir}")
os.makedirs(output_dir, exist_ok=True)
print(f"✅ Output directory '{output_dir}' is ready.\n")

📁 Creating output directory: ../data/ARIMA
✅ Output directory '../data/ARIMA' is ready.



In [13]:
# Process each stock
for stock_name, file_name in stocks.items():
    print(f"📊 Processing: {stock_name}")
    
    # Filter rows for the current stock
    stock_df = df[df['Stocks'] == stock_name][['Date', 'Close']].copy()
    
    if stock_df.empty:
        print(f"⚠️ No data found for {stock_name}. Skipping.\n")
        continue

    # Set date as index
    stock_df.set_index('Date', inplace=True)
    stock_df.sort_index(inplace=True)

    
    # ADF Test
    result = adfuller(stock_df['Close'])
    print(f"📉 ADF p-value for {stock_name}: {result[1]:.5f}")
    
    if result[1] > 0.05:
        # Not stationary, apply differencing
        stock_df['Close'] = stock_df['Close'].diff()
        stock_df.dropna(inplace=True)
        print(f"🔁 Applied differencing to make {stock_name} stationary.")

    # Save to CSV
    output_path = os.path.join(output_dir, f"{file_name}.csv")
    stock_df.to_csv(output_path)
    print(f"✅ Saved processed data for {stock_name} → {output_path}\n")

print("🎉 ARIMA transformation complete.")

📊 Processing: Apple
📉 ADF p-value for Apple: 0.99314
🔁 Applied differencing to make Apple stationary.
✅ Saved processed data for Apple → ../data/ARIMA\Apple.csv

📊 Processing: GeneralElectric
📉 ADF p-value for GeneralElectric: 0.99760
🔁 Applied differencing to make GeneralElectric stationary.
✅ Saved processed data for GeneralElectric → ../data/ARIMA\GeneralElectric.csv

📊 Processing: IBM
📉 ADF p-value for IBM: 0.99904
🔁 Applied differencing to make IBM stationary.
✅ Saved processed data for IBM → ../data/ARIMA\IBM.csv

📊 Processing: Johnson&Johnson
📉 ADF p-value for Johnson&Johnson: 0.93547
🔁 Applied differencing to make Johnson&Johnson stationary.
✅ Saved processed data for Johnson&Johnson → ../data/ARIMA\JohnsonJohnson.csv

📊 Processing: Microsoft
📉 ADF p-value for Microsoft: 1.00000
🔁 Applied differencing to make Microsoft stationary.
✅ Saved processed data for Microsoft → ../data/ARIMA\Microsoft.csv

🎉 ARIMA transformation complete.


## 📄 FINAL REPORT: ARIMA TRANSFORMATION

- Imported historical stock data for companies (Apple, IBM, Microsoft, etc.)
- Checked **stationarity** using the Augmented Dickey-Fuller (ADF) test
- Applied **first-order differencing** to convert non-stationary time series to stationary
- Saved transformed (stationary) data for each stock as CSV files
- Logged and printed ADF p-values to confirm transformation effectiveness